---
layout: post
codemirror: true
title: User API HW
categories: [Flask, Python, APIs]
lesson_language: Python
lesson_topic: User API HW
lesson_part: interactive
lesson_type: lesson
permalink: /python/flask/user-api-hw
author: rex0dex
---


## User API

Aaditya Prem (rex0dex). This lesson is about following one request all the way through my Flask backend: REST Client sends it, `api/user.py` checks it, `model/user.py` saves it, and a status code plus JSON comes back. All file names and line numbers below are from my OCS Flask fork, `usc-flask` (commit `34082c1`).

| Task | What I did |
| --- | --- |
| Authenticate with REST Client | Walked through `api/user_auth.http` and `GET /api/id`, and traced `_Security.post` and `auth_required` to see every status they can return |
| Read api/user.py | Found `user_api`, the `/api` prefix, the three registrations, and wrote the contract for `POST /api/user` |
| Follow model/user.py | Followed Create from the handler into `User.__init__`, `User.create`, `User.update`, and `User.read` |
| Inspect the CRUD request file | Read every step of `api/user_crud.http` and checked each expected status against the code |
| Trace and debug | Ran the whole sequence against a throwaway copy of my backend and recorded the call path for Update |

**Safety rules I followed:** I never put a real password in this notebook. The only requests sent to my running local server (port 8588) are read-only `GET`s with no login. Everything that logs in, creates, updates, or deletes runs against a temporary copy of the code with a brand new empty database, and that copy is deleted at the end.


## 1. Authenticate with REST Client

### The auth request file

`api/user_auth.http` is one request. The two `# @prompt` lines make REST Client ask me for my UID and password when I click **Send Request**, so neither one is ever saved in the file:

```http
### Authenticate
# @prompt uid Enter your user ID
# @prompt password Enter your password

POST {{baseUrl}}/api/authenticate
Content-Type: application/json
Accept: application/json

{
  "uid": "{{uid}}",
  "password": "{{password}}"
}
```

`{{baseUrl}}` comes from the REST Client environment in `.vscode/settings.json`. The `$shared` value is `http://127.0.0.1:8587`, `local` adds nothing on top of it, and `deployed` switches to `https://flask.opencodingsociety.com`. My local copy is on port 8588 today, so that is the port I use below.

### What the server does with it

`POST /api/authenticate` is handled by `UserAPI._Security.post` (`api/user.py` line 463):

1. Line 465: reads the JSON body. Empty body returns **400** `"Please provide user details"`.
2. Lines 473-478: missing `uid` or empty `password` returns **401** (`"User ID is missing"` / `"Password is missing"`).
3. Line 482: `User.query.filter_by(_uid=uid).first()` looks the user up.
4. Line 484: if there is no user, or `user.is_password(password)` (`model/user.py` line 304, `check_password_hash` on the stored pbkdf2 hash) is false, it returns **401** `"Invalid user id or password"`. It gives the same message for a wrong UID and a wrong password, so nobody can use it to find out which UIDs exist.
5. Line 495: `jwt.encode` signs a token with `_uid`, `token_version`, and `exp` (now + `JWT_TOKEN_MAX_AGE`, default 604800 seconds = 1 week) using `SECRET_KEY` and HS256.
6. Lines 508-517: success body is **200** with
   `{"message": "Authentication for <uid> successful", "user": {"uid", "name", "role", "class"}}`.
7. Lines 519-540: the token goes into a cookie named `jwt_python_flask` (`JWT_TOKEN_NAME`). Locally it is `SameSite=Lax`, not `Secure`, not `HttpOnly`. When `IS_PRODUCTION` is true it is `Secure`, `HttpOnly`, `SameSite=None`, domain `.opencodingsociety.com`.

The token is never in the JSON body, only in `Set-Cookie`. That is why REST Client matters here: it remembers response cookies per host, so every request after this one to the same host carries `jwt_python_flask` automatically. If I switch from `local` to `deployed`, that is a different host with no cookie, so I have to authenticate again.

The HTML `/login` route (`main.py` line 172) is a separate page with a form and a Flask-Login session. It is not this JSON endpoint.

### Checking who I am

```http
### Current identity - authenticate in user_auth.http first
GET {{baseUrl}}/api/id
Accept: application/json
```

This hits `UserAPI._ID.get` (`api/user.py` line 101). It has `@token_required()` on line 100, so before my code runs at all, `auth_required` in `api/authorize.py` decides whether I get in:

| Where it stops | Line(s) in `api/authorize.py` | Status | Message |
| --- | --- | --- | --- |
| No session and no cookie or `Authorization` header | 58-63 | 401 | Authentication required. No session or token found. |
| Token decodes but the user no longer exists | 70-75 | 401 | Invalid Authentication token! |
| Password changed after the token was made (`token_version` mismatch) | 80-85 | 401 | Token is no longer valid -- password has changed. |
| Token past its `exp` | 91-96 | 401 | Token has expired! |
| Token was tampered with or signed with another key | 97-102 | 401 | Invalid token! |
| Account is not `active` yet | 112-117 | 403 | Account pending verification. |
| Wrong role for the endpoint | 120-129 | 403 | Insufficient permissions. Required roles: ... |

If all of that passes, `g.current_user` is set (line 89) and `_ID.get` returns **200** with `current_user.read()` minus the password hash (`_without_password`, line 90), plus `role`, `is_admin`, and `is_teacher`.

So when `/api/id` fails I first ask **which boundary said no**. A 401/403 with one of the messages above came from `auth_required`, and the user model was never even reached.

Here is that first boundary on my running local server. These are plain `GET`s with no cookie, so nothing changes on the server:


In [1]:
# Read-only check against my running local backend: no login, nothing created or changed
import json, urllib.request, urllib.error

BASE = "http://localhost:8588"
for path in ["/api/id", "/api/user"]:
    req = urllib.request.Request(BASE + path, headers={"Accept": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=5) as r:
            print("GET", path, "->", r.status)
    except urllib.error.HTTPError as e:
        body = json.loads(e.read())
        print("GET", path, "->", e.code, body["error"], "|", body["message"])
    except urllib.error.URLError as e:
        print("GET", path, "-> server not running:", e.reason)


GET /api/id -> 401 Unauthorized | Authentication required. No session or token found.
GET /api/user -> 401 Unauthorized | Authentication required. No session or token found.


Both are **401** with the `auth_required` message from `api/authorize.py` lines 58-63. `/api/user` (the list) is also behind `@token_required()` (`api/user.py` line 257), so without a cookie I can't even see the user list.

### A throwaway copy for the requests that change data

To see the real responses for logging in and for the CRUD file without touching my real accounts, the next cell copies the backend code (skipping `instance/`, `.env`, `venv`, and `.git`) into a temp folder and imports it there. Flask puts the SQLite file under `app.instance_path`, so the copy gets its own brand new `instance/volumes/user_management.db`. With no `.env`, nothing like Kasm keys gets loaded either.

I make one test Admin with a random password from `secrets` that is never printed, and use Flask's test client, which keeps cookies between requests just like REST Client does.


In [2]:
# Throwaway copy of my backend with its own empty database
import os, sys, shutil, tempfile, secrets, io, contextlib, warnings
warnings.filterwarnings("ignore")   # SQLAlchemy relationship warnings, not part of this lesson

REPO = os.path.expanduser("~/rex0dex/usc-flask")
TMP = tempfile.mkdtemp(prefix="user_api_hw_")
COPY = os.path.join(TMP, "usc-flask")
shutil.copytree(REPO, COPY, ignore=shutil.ignore_patterns(
    "venv", ".git", "instance", ".env", "__pycache__", "*.log", "cookies.txt", "*.db"))
os.makedirs(os.path.join(COPY, "instance", "volumes"))
os.chdir(COPY)
sys.path.insert(0, COPY)

with contextlib.redirect_stdout(io.StringIO()):   # hide startup prints
    import main
from __init__ import app, db
from model.user import User

with app.app_context():
    db.create_all()
    db_file = db.engine.url.database
    print("database:", db_file.replace(TMP, "<temp>"))
    print("inside the temp folder:", db_file.startswith(TMP))
    print("users before I add any:", User.query.count())

    ADMIN_PW = secrets.token_urlsafe(16)            # random test password, never printed
    admin = User(name="HW Admin", uid="hw_admin", password=ADMIN_PW, role="Admin")
    admin.active = True                              # new accounts start inactive (model/user.py line 174)
    admin.create()
    print("test admin:", admin.uid, admin.role, "active =", admin.active)


database: <temp>/usc-flask/instance/volumes/user_management.db
inside the temp folder: True
users before I add any: 0
test admin: hw_admin Admin active = True


In [3]:
# Helper: send one request like REST Client and print status + a short JSON summary
KEYS = ["uid", "name", "role", "active", "verification_state", "access"]

def short(data):
    if isinstance(data, dict) and "uid" in data:
        return {k: data[k] for k in KEYS if k in data}
    if isinstance(data, list):
        return [short(d) for d in data]
    return data

def send(client, label, method, path, body=None):
    log = io.StringIO()
    with contextlib.redirect_stdout(log):           # _Security.post prints the token, so hide server prints
        resp = client.open(path, method=method, json=body, headers={"Accept": "application/json"})
    print(f"{label}: {method} {path} -> {resp.status_code}")
    if resp.data:
        print("   ", short(resp.get_json()))
    else:
        print("    (empty body)")
    return resp

rest = app.test_client()   # keeps cookies per host, like REST Client

# Failures first, then the real login
send(rest, "no cookie yet", "GET", "/api/id")
send(rest, "empty body", "POST", "/api/authenticate", {})
send(rest, "no password", "POST", "/api/authenticate", {"uid": "hw_admin"})
send(rest, "wrong password", "POST", "/api/authenticate", {"uid": "hw_admin", "password": "not-the-password"})
send(rest, "unknown uid", "POST", "/api/authenticate", {"uid": "nobody_here", "password": "not-the-password"})
r = send(rest, "user_auth.http", "POST", "/api/authenticate", {"uid": "hw_admin", "password": ADMIN_PW})

cookie = r.headers["Set-Cookie"]
name, _, rest_of_cookie = cookie.partition("=")
print("    Set-Cookie name:", name, "| attributes:", "; ".join(rest_of_cookie.split("; ")[1:]))
print("    token in JSON body?", "jwt" in json.dumps(r.get_json()).lower())

r = send(rest, "identity", "GET", "/api/id")
print("    is_admin:", r.get_json()["is_admin"], "| password in response?", "password" in r.get_json())


no cookie yet: GET /api/id -> 401
    {'message': 'Authentication required. No session or token found.', 'data': None, 'error': 'Unauthorized'}
empty body: POST /api/authenticate -> 400
    {'message': 'Please provide user details', 'data': None, 'error': 'Bad request'}
no password: POST /api/authenticate -> 401
    {'message': 'Password is missing'}
wrong password: POST /api/authenticate -> 401
    {'message': 'Invalid user id or password'}
unknown uid: POST /api/authenticate -> 401
    {'message': 'Invalid user id or password'}
user_auth.http: POST /api/authenticate -> 200
    {'message': 'Authentication for hw_admin successful', 'user': {'class': [], 'name': 'HW Admin', 'role': 'Admin', 'uid': 'hw_admin'}}
    Set-Cookie name: jwt_python_flask | attributes: Expires=Mon, 16 Nov 2026 19:06:33 GMT; Max-Age=604800; Path=/; SameSite=Lax
    token in JSON body? False
identity: GET /api/id -> 200
    {'uid': 'hw_admin', 'name': 'HW Admin', 'role': 'Admin', 'active': True, 'verification_sta

What this shows:

* Before login, `/api/id` is stopped by `auth_required` (401), same as on my real server.
* An empty body is **400**, but a missing password, a wrong password, and an unknown UID are all **401**. The wrong password and the unknown UID give the exact same message.
* The successful login is **200**, the JSON only has `message` and `user`, and the token only shows up in `Set-Cookie` as `jwt_python_flask` with `Path=/` and `SameSite=Lax` (the local branch, lines 531-540).
* After that, `/api/id` returns my test Admin with `is_admin: True`, and `_without_password` removed the hash.

## 2. Read api/user.py

* `user_api = Blueprint('user_api', __name__, url_prefix='/api')` is on lines 17-18, and `api = Api(user_api)` is on line 21. `main.py` line 86 registers the blueprint, so every resource path below starts with `/api`.
* Registrations: `api.add_resource(_ID, '/id')` line 914, `api.add_resource(_CRUD, '/user')` line 916, `api.add_resource(_Security, '/authenticate')` line 919. Verify is line 924, `'/user/<string:uid>/verify'`.
* Classes live inside `class UserAPI` (line 98): `_ID` line 99, `_CRUD` line 152, `_Security` line 462.

How `_CRUD` maps HTTP methods to Python methods, and what guards each one:

| Request | Method in `_CRUD` | Decorator | Success |
| --- | --- | --- | --- |
| `POST /api/user` | `post` line 153 | **none** | 200 + user JSON |
| `GET /api/user` | `get` line 258 | `@token_required()` line 257 | 200 + list (or `users` + `pagination` if `?page=` is given) |
| `PUT /api/user` | `put` line 319 | `@token_required()` line 318 | 200 + updated user |
| `DELETE /api/user` | `delete` line 362 | `@token_required("Admin")` line 361 | 204, no body |

### My contract: `POST /api/user` (Create)

* **Method and path:** `POST /api/user`, JSON body.
* **Required inputs:** `name` at least 2 characters (lines 170-172), `uid` at least 2 characters (lines 175-177).
* **Optional inputs:** `password` (at least 8 characters, unless it is already a `pbkdf2:sha256:` hash, lines 195-198), `accountType` (looked up with `account_types.get` on line 188, falls back to `student`), `email`, `sid`, `school`, `kasm_server_needed`, `class`, `game_profile`, and `idToken` for Google. `dob` is ignored.
* **Authentication:** none. There is no decorator on `post`, because this is also the signup endpoint. The new account's `role` comes from `model/account_types.json` (`mentor` -> `Mentor`, `student` -> `User`), so a caller can't just send `"role": "Admin"`. A Mentor starts inactive until a Teacher or Admin verifies it.
* **Success:** **200**, not 201. Line 251 is `return jsonify(_without_password(user.read()))`, and `jsonify` defaults to 200.
* **Failures:** 400 bad name, 400 bad uid, **409** if the UID is taken (lines 181-182, this fork accepts any unused username instead of checking GitHub), 400 short password, 400 if `create` returns `None` and the user isn't in the DB (line 244), 500 if anything throws (lines 253-255).

## 3. Follow model/user.py

The handler gets the model from `from model.user import User` on `api/user.py` line 11. For Create, the call boundary goes like this:

1. **API builds the object.** Lines 199-201 call `User(name=..., uid=..., password=..., role=role)`, which is `User.__init__` (`model/user.py` line 189). It calls `self.set_password(password)` (line 288), which hashes the password with `pbkdf2:sha256` and bumps `token_version`. Nothing is saved yet.
2. **API cleans the body.** Lines 204-228 build `cleaned_body` with only the fields the model knows about and drop `None` values.
3. **Model saves it.** Line 233 `user_obj.create(cleaned_body)` goes to `User.create` (line 518): `db.session.add(self)`, `db.session.commit()`, then `self.update(inputs)`. An `IntegrityError` (like a duplicate `_uid`, which is `unique=True` on line 150) rolls back and returns `None`.
4. **Model updates the extra fields.** `User.update` (line 559) copies fields over, only talks to Kasm if `kasm_server_needed` is true, and commits again on line 637.
5. **Back to the API.** `User.read` (line 531) turns the object into a dict, which still has `"password"`. The API strips it with `_without_password` and returns it with `jsonify`.

Who is responsible for what:

| Responsibility | Where |
| --- | --- |
| Authentication (who are you) | `_Security.post`, `User.is_password` (line 304), `auth_required` in `api/authorize.py` |
| Authorization (are you allowed) | `@token_required(...)` roles, `User.deletion_blocker` (line 362), the Admin branch in `_CRUD.put` (lines 335-344) |
| API validation (is the input OK) | the `if` checks at the top of each `_CRUD` method, returning 400/404/409 |
| Persistence (save it) | `User.create`, `User.update`, `User.delete` (line 645), each with `db.session.commit()` and `rollback()` on `IntegrityError` |

## 4. Inspect the CRUD request file

`api/user_crud.http` sets `@testUid = wilma` and then has 6 requests. I checked each expected status in its comments against the code:

| Step | Request | File expects | Code that decides it |
| --- | --- | --- | --- |
| 1 | `POST /api/user` (name, uid, dob, accountType mentor, kasm) | 200 | `_CRUD.post`, `jsonify` on line 251 (200, not 201) |
| 1b | `POST /api/user/wilma/verify` | 200, `"active": true` | `_VerifyAccount.post` line 873, `@token_required(["Teacher", "Admin"])`, calls `User.activate` (line 336) |
| 2 | `GET /api/id` | 200, role Admin | `_ID.get` returns `g.current_user`, which is whoever owns the cookie, not wilma |
| 3 | `GET /api/user` | 200, array with wilma | `_CRUD.get`, Admin gets `access: ['rw']` on every user (lines 297-298) |
| 4 | `PUT /api/user` with uid + new name | 200, new name | `_CRUD.put`: Admin + different uid goes to lines 340-344 to find wilma, then `user.update(body)` line 356 |
| 5 | `DELETE /api/user` | 204, no body | `_CRUD.delete` returns a string with 204 (line 390), and a 204 response has no body, so there is nothing to parse |

Things I noticed before running it:

* The file has `"kasm_server_needed": true`. The lesson says to set it to `false` unless I really want a Kasm server, so I change that in my run.
* Steps 1b-5 need an Admin cookie. Step 1 doesn't (no decorator), but the file still says to authenticate first because the rest of the sequence fails without it.
* If Create fails I stop. Update and Delete use `testUid`, so if Create failed because `wilma` already existed, the next steps would rename and delete someone else's account.

## 5. Trace and debug

Running the CRUD file one request at a time on the throwaway copy, with the Admin cookie from above:


In [4]:
# api/user_crud.http, steps 1 through 3 (kasm_server_needed set to false)
create_body = {"uid": "wilma", "name": "Wilma Flinstone", "dob": "1000-01-02",
               "accountType": "mentor", "kasm_server_needed": False}

r = send(rest, "1. create", "POST", "/api/user", create_body)
if r.status_code != 200:
    raise SystemExit("Create failed, stopping before update/delete")

send(rest, "1 again", "POST", "/api/user", create_body)                     # expected failure
send(rest, "short name", "POST", "/api/user", {"uid": "pebbles", "name": "P"})  # expected failure
send(rest, "1b. verify", "POST", "/api/user/wilma/verify")
send(rest, "2. current user", "GET", "/api/id")
send(rest, "3. list users", "GET", "/api/user")


1. create: POST /api/user -> 200
    {'uid': 'wilma', 'name': 'Wilma Flinstone', 'role': 'Mentor', 'active': False, 'verification_state': 'inactive'}
1 again: POST /api/user -> 409
    {'message': 'The username wilma is already taken. Please choose another.'}
short name: POST /api/user -> 400
    {'message': 'Name is missing, or is less than 2 characters'}
1b. verify: POST /api/user/wilma/verify -> 200
    {'uid': 'wilma', 'name': 'Wilma Flinstone', 'role': 'Mentor', 'active': True, 'verification_state': 'active_verified'}
2. current user: GET /api/id -> 200
    {'uid': 'hw_admin', 'name': 'HW Admin', 'role': 'Admin', 'active': True, 'verification_state': 'active_verification_required'}
3. list users: GET /api/user -> 200
    [{'uid': 'hw_admin', 'name': 'HW Admin', 'role': 'Admin', 'active': True, 'verification_state': 'active_verification_required', 'access': ['rw']}, {'uid': 'wilma', 'name': 'Wilma Flinstone', 'role': 'Mentor', 'active': True, 'verification_state': 'active_verified'

Every status matches the comments in the file and my contract: Create is **200** with `active: False` and `role: Mentor`, sending it again is **409** from line 182, a 1-letter name is **400** from line 172, verify flips `active` to `True`, `/api/id` is still `hw_admin`, and the list shows both users with `rw` access.

### Breakpoints and the Update path

In VS Code the breakpoints I'd use are `api/authorize.py` line 67 (`jwt.decode`), `api/user.py` line 332 (`body = request.get_json()` in `put`), and `model/user.py` line 559 (`User.update`). Then send step 4 from REST Client, look at `body` and `current_user`, step into `user.update(body)`, and press Continue, because REST Client just waits until the paused request finishes.

To record the same path as evidence, this cell turns on Python's profiler hook for one request and prints every function it enters inside the three files I care about. The number is where Python says the function starts, which includes its decorator line (so `put` shows 318, the `@token_required()` line, not 319):


In [5]:
# Step 4 (Update) with a call trace through authorize.py -> api/user.py -> model/user.py
WATCH = {os.path.join(COPY, p): p for p in ["api/authorize.py", "api/user.py", "model/user.py"]}
calls = []

def tracer(frame, event, arg):
    if event == "call" and frame.f_code.co_filename in WATCH:
        step = f"{WATCH[frame.f_code.co_filename]}:{frame.f_code.co_firstlineno} {frame.f_code.co_name}"
        if not calls or calls[-1] != step:
            calls.append(step)

sys.setprofile(tracer)
r = send(rest, "4. update", "PUT", "/api/user", {"uid": "wilma", "name": "Wilma Flintstone Updated"})
sys.setprofile(None)

seen = set()
for step in calls:
    if step not in seen:          # first time each function is entered
        seen.add(step)
        print("   ", step)


4. update: PUT /api/user -> 200
    {'uid': 'wilma', 'name': 'Wilma Flintstone Updated', 'role': 'Mentor', 'active': True, 'verification_state': 'active_verified'}
    api/authorize.py:36 decorated
    api/user.py:318 put
    model/user.py:314 role
    model/user.py:260 uid
    model/user.py:559 update
    model/user.py:255 name
    model/user.py:789 set_uid
    model/user.py:531 read
    model/user.py:250 name
    model/user.py:229 email
    model/user.py:270 sid
    model/user.py:383 pfp
    model/user.py:392 grade_data
    model/user.py:404 ap_exam
    model/user.py:416 school
    model/user.py:424 game_profile
    model/user.py:328 verification_state
    model/user.py:711 read_sections
    model/user.py:724 read_personas
    api/user.py:90 _without_password


Reading the trace top to bottom: `decorated` in `authorize.py` runs first (the `@token_required()` wrapper), and only then my `put`. `put` reads `current_user.role` and `current_user.uid` (the Admin branch), then calls `update` (line 559). Inside `update`, the `name` setter runs, and `set_uid` (line 789) does nothing because the uid didn't change. Then `put` calls `read` (line 531), which goes through every property getter plus `read_sections` and `read_personas`, and last `_without_password` strips the hash. The Update response is **200** with the new name, and nothing in the trace touches GitHub because the uid stayed the same (lines 349-352).

### One expected failure: a pending account

To see a request fail at the right boundary, I make a second Mentor, `barney`, with a random password and log in as him in a separate client. He never gets verified:


In [6]:
# Expected failure: login works, but every protected endpoint refuses an inactive account
BARNEY_PW = secrets.token_urlsafe(16)
send(rest, "create barney", "POST", "/api/user",
     {"uid": "barney", "name": "Barney Rubble", "password": BARNEY_PW,
      "accountType": "mentor", "kasm_server_needed": False})

barney = app.test_client()
send(barney, "barney login", "POST", "/api/authenticate", {"uid": "barney", "password": BARNEY_PW})
send(barney, "barney identity", "GET", "/api/id")
send(barney, "barney delete", "DELETE", "/api/user", {"uid": "wilma"})


create barney: POST /api/user -> 200
    {'uid': 'barney', 'name': 'Barney Rubble', 'role': 'Mentor', 'active': False, 'verification_state': 'inactive'}
barney login: POST /api/authenticate -> 200
    {'message': 'Authentication for barney successful', 'user': {'class': [], 'name': 'Barney Rubble', 'role': 'Mentor', 'uid': 'barney'}}
barney identity: GET /api/id -> 403
    {'message': 'Account pending verification.', 'data': None, 'error': 'Forbidden'}
barney delete: DELETE /api/user -> 403
    {'message': 'Account pending verification.', 'data': None, 'error': 'Forbidden'}


Barney's login is **200**: `_Security.post` only checks the password, not `active`. But `/api/id` and the delete are both **403** "Account pending verification." from `api/authorize.py` lines 112-117. That check comes before the role check on line 120, so the delete never even gets to "you need Admin". This is the lesson's point about boundaries: the login worked, the user model is fine, and the request was refused by the auth decorator because `active` is `False`.

### Step 5 (Delete) and the cleanup


In [7]:
# api/user_crud.http step 5, then two expected failures
r = send(rest, "5. delete", "DELETE", "/api/user", {"uid": "wilma"})
print("    bytes in body:", len(r.data))
send(rest, "delete again", "DELETE", "/api/user", {"uid": "wilma"})
send(rest, "delete myself", "DELETE", "/api/user", {"uid": "hw_admin"})
send(rest, "cleanup barney", "DELETE", "/api/user", {"uid": "barney"})

with app.app_context():
    print("users left:", [u.uid for u in User.query.all()])
    db.session.remove()
    db.engine.dispose()

os.chdir(REPO)
sys.path.remove(COPY)
shutil.rmtree(TMP)
print("temp copy removed:", not os.path.exists(TMP))


5. delete: DELETE /api/user -> 204
    (empty body)
    bytes in body: 0
delete again: DELETE /api/user -> 404
    {'message': 'User wilma not found'}
delete myself: DELETE /api/user -> 403
    {'message': 'You cannot delete your own account; another Admin must do it'}
cleanup barney: DELETE /api/user -> 204
    (empty body)
users left: ['hw_admin']
temp copy removed: True


Delete is **204** with a 0-byte body, so a client that calls `.json()` on it would crash. That's why the file says not to parse it. Deleting `wilma` again is **404** from line 379 (the `User.query` on line 375 found nothing). Deleting myself is **403** from `User.deletion_blocker` (`model/user.py` lines 366-367), which blocks an Admin from deleting their own account, even though the decorator already let me in as Admin. Then I delete barney and remove the temp folder.

### Explaining one success and one failure

**Success, step 1 Create.** `POST /api/user` has no decorator, so `_CRUD.post` runs right away. `wilma` passes the name and uid checks, isn't taken (line 181), and `account_types.get("mentor")` gives role `Mentor` with no Google domains, so there's no `verified_email`. `User(...)` builds the object and hashes the default password. `User.create` commits it and `User.update` saves the extra fields. Since `verified_email` is `None`, line 247 doesn't activate it. The response is 200 with `active: False` and `verification_state: inactive`, which is exactly what the CRUD file says step 1b is for.

**Expected failure, barney's `/api/id`.** The 403 has `"error": "Forbidden"` and "Account pending verification.", which only shows up in `api/authorize.py` line 114. His cookie was valid (the decode on line 67 worked and his `token_version` matched), so the token isn't the problem. The account is just inactive. The fix is step 1b (`/api/user/barney/verify` by a Teacher or Admin), not turning off the check.

### Frontend and CORS

REST Client and the test client send requests directly, so they never do a CORS preflight. A browser frontend on another origin does: it sends `OPTIONS` first, and it only gets the cookie through if its origin is in `allowed_origins` (`__init__.py` lines 33-48, for example `http://127.0.0.1:4000` and `https://rex0dex.github.io`) with `supports_credentials=True` (line 52). `auth_required` also lets `OPTIONS` through with a 200 (`api/authorize.py` lines 132-133). So if my frontend fails but REST Client works, I compare the browser's Network tab (method, URL, body, whether the `Cookie` header was sent) to the working REST Client request. The difference is usually the origin or `credentials: 'include'`, not the API code.

## Takeaways

* The JWT only lives in the `jwt_python_flask` cookie, and REST Client resends it for me on the same host.
* 401 means "who are you?" and 403 means "I know who you are, but no". Both can come from `auth_required` before my handler ever runs.
* Create returns 200 in this API, Delete returns 204 with no body, and a duplicate UID is 409.
* `api/user.py` validates and checks permissions, `model/user.py` saves (`create`, `update`, `delete` with commit/rollback), and `_without_password` keeps the hash out of every response.
